# Gaussian Mixture Regimes with Robustness Diagnostics

Uses Gaussian mixture regimes and reports walk-forward behavior.

This notebook downloads daily historical market prices from Yahoo Finance. Results depend on the configured dates and symbols.


In [ ]:
"""
GM-REGIME MULTI-ASSET STRATEGY - PRODUCTION IMPLEMENTATION
==========================================================
Asset rotation based on volatility regimes:
- Low Vol: QQQ (Growth/Tech)
- Transition: SPY (Market)
- High Vol: SH (Inverse S&P)
"""

from _market_data import download_price_records
import pandas as pd
import numpy as np
from arch import arch_model
from scipy.special import softmax
from scipy.optimize import differential_evolution
from scipy.stats import norm, skew, kurtosis
from typing import Dict, Tuple, List
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec
import seaborn as sns
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")


class Config:
    ASSET_BULL = "QQQ"
    ASSET_NEUTRAL = "SPY"
    ASSET_BEAR = "SH"
    BENCHMARK_SYMBOL = "URTH"
    START_DATE = "2013-12-31"
    END_DATE = "2021-12-31"
    WARMUP_DAYS = 252
    TRAIN_DAYS = 252
    VALIDATE_DAYS = 126
    TEST_DAYS = 126
    INITIAL_CAPITAL = 100000.0
    COMPOUNDING_MODE = True
    LEVERAGE = 1.0
    COMMISSION_PER_SHARE = 0.005
    COMMISSION_MIN = 1.0
    COMMISSION_MAX = 0.005
    SLIPPAGE_BPS = 5.0
    SWITCH_COST_MULTIPLIER = 2.0
    YZ_WINDOW = 5
    YZ_MEDIAN_WINDOW = 60
    HAR_DAILY_LAG = 1
    HAR_WEEKLY_LAG = 5
    HAR_MONTHLY_LAG = 22
    GARCH_P = 1
    GARCH_Q = 1
    GARCH_MIN_WINDOW = 100
    LOW_VOL_THRESHOLD = 0.3
    HIGH_VOL_THRESHOLD = 0.7
    REGIME_PIVOT = 0.01
    REGIME_SCALE = 100.0
    MIN_GRI = 0.65


def get_fmp_data(symbol: str, start: str, end: str) -> pd.DataFrame:
    """Fetch OHLCV data from Yahoo Finance."""
    response_prices = {"historical": download_price_records(symbol, start, end)}
    data = response_prices
    df = pd.DataFrame(data["historical"])
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").set_index("date")
    df = df.rename(columns={"adjClose": "adj_close"})
    required_cols = ["open", "high", "low", "close", "adj_close", "volume"]
    for col in required_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df[required_cols].dropna()


def get_multi_asset_data(tickers: List[str], start: str, end: str) -> Dict[str, pd.DataFrame]:
    """
    Download and synchronize data for multiple assets.
    Returns dict with aligned DataFrames (common date index).
    """
    data_dict = {}
    for ticker in tickers:
        print(f"  Downloading {ticker}...")
        data_dict[ticker] = get_fmp_data(ticker, start, end)
    common_idx = data_dict[tickers[0]].index
    for ticker in tickers[1:]:
        common_idx = common_idx.intersection(data_dict[ticker].index)
    aligned_data = {ticker: df.loc[common_idx] for ticker, df in data_dict.items()}
    print(f"  ✓ Aligned to {len(common_idx)} common dates")
    return aligned_data


def compute_yang_zhang_volatility(df: pd.DataFrame, window: int = 5) -> pd.Series:
    """Calculate Yang-Zhang volatility estimator."""
    log_ho = np.log(df["high"] / df["open"])
    log_lo = np.log(df["low"] / df["open"])
    log_co = np.log(df["close"] / df["open"])
    log_oc = np.log(df["open"] / df["close"].shift(1))
    log_oo = np.log(df["open"] / df["open"].shift(1))
    rs_var = (log_ho * (log_ho - log_co) + log_lo * (log_lo - log_co)).rolling(window=window).mean()
    oc_var = log_oc.rolling(window=window).var()
    oo_var = log_oo.rolling(window=window).var()
    k = 0.34 / (1.34 + (window + 1) / (window - 1))
    yz_var = oo_var + k * oc_var + (1 - k) * rs_var
    return np.sqrt(yz_var.clip(lower=0))


def compute_har_features(yz_vol: pd.Series) -> pd.DataFrame:
    """Compute HAR features from realized volatility."""
    features = pd.DataFrame(index=yz_vol.index)
    features["rv_daily"] = yz_vol.shift(Config.HAR_DAILY_LAG)
    features["rv_weekly"] = yz_vol.rolling(window=Config.HAR_WEEKLY_LAG).mean().shift(1)
    features["rv_monthly"] = yz_vol.rolling(window=Config.HAR_MONTHLY_LAG).mean().shift(1)
    yz_median = yz_vol.rolling(window=Config.YZ_MEDIAN_WINDOW).median()
    features["rel_vol"] = (yz_vol / yz_median - 1.0).shift(1)
    return features


def fit_har_garch_x(df: pd.DataFrame, p: int = 1, q: int = 1, rescale: float = 100.0) -> np.ndarray:
    """Fit GARCH with HAR features using expanding window."""
    try:
        yz_vol = compute_yang_zhang_volatility(df, window=Config.YZ_WINDOW)
        har_features = compute_har_features(yz_vol)
        returns = np.log(df["adj_close"] / df["adj_close"].shift(1)) * rescale
        valid_idx = returns.index.intersection(har_features.dropna().index)
        model_data = pd.DataFrame(
            {
                "returns": returns.loc[valid_idx],
                "rv_daily": har_features.loc[valid_idx, "rv_daily"] * rescale,
                "rv_weekly": har_features.loc[valid_idx, "rv_weekly"] * rescale,
                "rv_monthly": har_features.loc[valid_idx, "rv_monthly"] * rescale,
                "rel_vol": har_features.loc[valid_idx, "rel_vol"],
            }
        )
        sigma_estimates = np.full(len(df), np.nan)
        date_to_idx = {date: i for i, date in enumerate(df.index)}
        min_window = max(Config.GARCH_MIN_WINDOW, Config.HAR_MONTHLY_LAG + 50)
        for i in range(min_window, len(model_data)):
            history = model_data.iloc[:i]
            try:
                X_vars = history[["rv_daily", "rv_weekly", "rv_monthly"]].dropna()
                y_returns = history["returns"].loc[X_vars.index]
                if len(y_returns) < 50:
                    continue
                model = arch_model(
                    y_returns, x=X_vars, vol="GARCH", p=p, q=q, dist="t", rescale=False
                )
                result = model.fit(disp="off", options={"maxiter": 100}, show_warning=False)
                forecast_x = model_data.iloc[i][
                    ["rv_daily", "rv_weekly", "rv_monthly"]
                ].values.reshape(1, -1)
                forecasts = result.forecast(horizon=1, x=forecast_x, reindex=False)
                next_vol = np.sqrt(forecasts.variance.values[-1, 0])
                current_date = model_data.index[i]
                if current_date in date_to_idx:
                    sigma_estimates[date_to_idx[current_date]] = next_vol / rescale
            except:
                current_date = model_data.index[i]
                if current_date in date_to_idx:
                    sigma_estimates[date_to_idx[current_date]] = (
                        history["rv_daily"].iloc[-1] / rescale
                    )
        mask = np.isnan(sigma_estimates)
        if mask.any():
            proxy = returns.rolling(20, min_periods=10).std().values
            sigma_estimates[mask] = proxy[mask]
        sigma_series = pd.Series(sigma_estimates, index=df.index)
        sigma_series = sigma_series.ffill().bfill()
        return sigma_series.values
    except Exception as e:
        print(f"HAR-GARCH-X failed: {e}. Using fallback.")
        returns = np.log(df["adj_close"] / df["adj_close"].shift(1))
        return returns.rolling(20, min_periods=10).std().bfill().values


def compute_regime_probabilities(
    sigma: np.ndarray, a1: float = -3.0, a2: float = 3.0
) -> pd.DataFrame:
    """Convert volatility to regime probabilities."""
    centered_input = (sigma - Config.REGIME_PIVOT) * Config.REGIME_SCALE
    logits = np.vstack([a1 * centered_input, a2 * centered_input])
    probs = softmax(logits, axis=0)
    return pd.DataFrame({"p_low": probs[0], "p_high": probs[1]})


def generate_regime_signals(
    df: pd.DataFrame,
    low_threshold: float,
    high_threshold: float,
    a1: float = -3.0,
    a2: float = 3.0,
    precomputed_sigma: np.ndarray = None,
) -> pd.DataFrame:
    """
    Generate regime signals with target asset allocation.

    Returns DataFrame with:
    - regime: 0 (Low Vol), 1 (Transition), 2 (High Vol)
    - target_ticker: QQQ, SPY, or SH
    """
    if precomputed_sigma is None:
        sigma = fit_har_garch_x(df, Config.GARCH_P, Config.GARCH_Q)
    else:
        sigma = precomputed_sigma
    probs = compute_regime_probabilities(sigma, a1, a2)
    probs.index = df.index
    regime = np.full(len(probs), 1)
    regime[probs["p_high"] < low_threshold] = 0
    regime[probs["p_high"] >= high_threshold] = 2
    probs["regime"] = regime
    asset_map = {0: Config.ASSET_BULL, 1: Config.ASSET_NEUTRAL, 2: Config.ASSET_BEAR}
    probs["target_ticker"] = probs["regime"].map(asset_map)
    return probs


def calculate_commission(shares: float, price: float) -> float:
    """IBKR-style tiered commission."""
    if shares == 0:
        return 0.0
    base_commission = abs(shares) * Config.COMMISSION_PER_SHARE
    trade_value = abs(shares * price)
    max_commission = trade_value * Config.COMMISSION_MAX
    return max(Config.COMMISSION_MIN, min(base_commission, max_commission))


def calculate_slippage(shares: float, price: float) -> float:
    """Market impact slippage."""
    return abs(shares * price) * (Config.SLIPPAGE_BPS / 10000.0)


def calculate_total_costs(shares: float, price: float, is_switch: bool = False) -> float:
    """
    Total transaction costs.
    is_switch=True applies multiplier for asset rotation (sell + buy).
    """
    base_cost = calculate_commission(shares, price) + calculate_slippage(shares, price)
    return base_cost * Config.SWITCH_COST_MULTIPLIER if is_switch else base_cost


class MultiAssetBacktestEngine:
    """Backtest engine with asset rotation capability."""

    def __init__(self, price_data_dict: Dict[str, pd.DataFrame], signals: pd.DataFrame):
        self.price_data = price_data_dict
        self.signals = signals.shift(1)
        self.initial_capital = Config.INITIAL_CAPITAL
        self.cash = Config.INITIAL_CAPITAL
        self.position_shares = 0.0
        self.current_ticker = None
        self.portfolio_value = Config.INITIAL_CAPITAL
        self.equity_curve = []
        self.common_dates = self.price_data[list(self.price_data.keys())[0]].index

    def run(self) -> pd.DataFrame:
        """Execute multi-asset backtest with asset rotation."""
        dates = self.common_dates
        for i, date in enumerate(dates):
            if i == 0:
                self.equity_curve.append(
                    {
                        "date": date,
                        "portfolio_value": self.portfolio_value,
                        "daily_return": 0.0,
                        "ticker": self.current_ticker,
                    }
                )
                continue
            prev_date = dates[i - 1]
            if self.current_ticker is not None:
                close_prev = self.price_data[self.current_ticker].loc[prev_date, "adj_close"]
                prev_portfolio_close = self.cash + self.position_shares * close_prev
            else:
                prev_portfolio_close = self.cash
            has_signal = date in self.signals.index
            if has_signal and (not pd.isna(self.signals.loc[date, "target_ticker"])):
                target_ticker = self.signals.loc[date, "target_ticker"]
                if target_ticker != self.current_ticker:
                    if self.current_ticker is not None and self.position_shares > 0:
                        sell_price = self.price_data[self.current_ticker].loc[date, "open"]
                        sell_value = self.position_shares * sell_price
                        sell_costs = calculate_total_costs(
                            self.position_shares, sell_price, is_switch=True
                        )
                        self.cash += sell_value - sell_costs
                        self.position_shares = 0.0
                    buy_price = self.price_data[target_ticker].loc[date, "open"]
                    if Config.COMPOUNDING_MODE:
                        sizing_base = prev_portfolio_close
                    else:
                        sizing_base = self.initial_capital
                    target_notional_gross = sizing_base * Config.LEVERAGE
                    approx_shares = target_notional_gross / buy_price if buy_price > 0 else 0
                    is_switch_trade = self.current_ticker is not None
                    estimated_costs = calculate_total_costs(
                        approx_shares, buy_price, is_switch=is_switch_trade
                    )
                    target_notional_net = target_notional_gross - estimated_costs
                    target_shares = target_notional_net / buy_price if buy_price > 0 else 0
                    buy_value = target_shares * buy_price
                    actual_costs = calculate_total_costs(
                        target_shares, buy_price, is_switch=is_switch_trade
                    )
                    total_required = buy_value + actual_costs
                    if total_required <= self.cash and target_shares > 0:
                        self.cash -= total_required
                        self.position_shares = target_shares
                        self.current_ticker = target_ticker
            if self.current_ticker is not None:
                close_t = self.price_data[self.current_ticker].loc[date, "adj_close"]
                self.portfolio_value = self.cash + self.position_shares * close_t
            else:
                self.portfolio_value = self.cash
            daily_return = (
                self.portfolio_value / prev_portfolio_close - 1.0
                if prev_portfolio_close > 0
                else 0.0
            )
            self.equity_curve.append(
                {
                    "date": date,
                    "portfolio_value": self.portfolio_value,
                    "daily_return": daily_return,
                    "ticker": self.current_ticker,
                }
            )
        return pd.DataFrame(self.equity_curve).set_index("date")


def calculate_performance_metrics(returns: pd.Series, label: str = "Strategy") -> Dict:
    """Calculate comprehensive performance metrics."""
    returns = returns.dropna()
    if len(returns) == 0:
        return {}
    total_return = (1 + returns).prod() - 1
    years = len(returns) / 252.0
    annual_return = (1 + total_return) ** (1 / years) - 1 if years > 0 else 0.0
    annual_vol = returns.std() * np.sqrt(252)
    sharpe = annual_return / annual_vol if annual_vol > 0 else 0.0
    cumulative = (1 + returns).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    max_drawdown = drawdown.min()
    calmar = annual_return / abs(max_drawdown) if max_drawdown != 0 else 0.0
    win_rate = (returns > 0).sum() / len(returns)
    return {
        "label": label,
        "total_return": total_return,
        "annual_return": annual_return,
        "annual_volatility": annual_vol,
        "sharpe_ratio": sharpe,
        "max_drawdown": max_drawdown,
        "calmar_ratio": calmar,
        "win_rate": win_rate,
        "skewness": skew(returns),
        "kurtosis": kurtosis(returns),
        "num_periods": len(returns),
    }


def optimize_parameters_in_sample(
    train_data: pd.DataFrame, validate_data_dict: Dict[str, pd.DataFrame]
) -> Dict:
    """
    Optimize regime parameters using SPY as volatility reference.

    Args:
        train_data: Historical SPY data (not used in current implementation)
        validate_data_dict: {ticker: DataFrame} for all assets in validation period
    """
    spy_data = validate_data_dict[Config.ASSET_NEUTRAL]
    print("  Pre-calculating HAR-GARCH-X volatilities...")
    sigma_precomputed = fit_har_garch_x(spy_data, Config.GARCH_P, Config.GARCH_Q)
    print("  ✓ Volatility pre-calculation complete")

    def objective(params):
        low_thresh, high_thresh, a1, a2 = params
        if low_thresh >= high_thresh:
            return 1000000.0
        try:
            signals = generate_regime_signals(
                spy_data, low_thresh, high_thresh, a1, a2, precomputed_sigma=sigma_precomputed
            )
            engine = MultiAssetBacktestEngine(validate_data_dict, signals)
            results = engine.run()
            val_sharpe = (
                results["daily_return"].mean() / results["daily_return"].std() * np.sqrt(252)
            )
            return -val_sharpe
        except:
            return 1000000.0

    bounds = [(0.2, 0.4), (0.6, 0.8), (-5.0, -1.0), (1.0, 5.0)]
    result = differential_evolution(
        objective, bounds=bounds, maxiter=20, seed=42, workers=1, polish=False
    )
    return {
        "low_threshold": result.x[0],
        "high_threshold": result.x[1],
        "a1": result.x[2],
        "a2": result.x[3],
        "val_sharpe": -result.fun,
    }


def walk_forward_analysis(
    price_data_dict: Dict[str, pd.DataFrame],
) -> Tuple[pd.DataFrame, pd.Series]:
    """Execute walk-forward validation for multi-asset strategy."""
    spy_data = price_data_dict[Config.ASSET_NEUTRAL]
    dates = spy_data.index
    total_days = len(dates)
    min_required = Config.WARMUP_DAYS + Config.TRAIN_DAYS + Config.VALIDATE_DAYS + Config.TEST_DAYS
    if total_days < min_required:
        raise ValueError(f"Insufficient data: need {min_required}, have {total_days}")
    fold_results = []
    all_test_returns = []
    all_test_dates = []
    fold_num = 0
    current_start = Config.WARMUP_DAYS
    while current_start + Config.TRAIN_DAYS + Config.VALIDATE_DAYS + Config.TEST_DAYS <= total_days:
        fold_num += 1
        train_start = 0
        train_end = current_start + Config.TRAIN_DAYS
        val_end = train_end + Config.VALIDATE_DAYS
        test_end = val_end + Config.TEST_DAYS
        train_dates = dates[train_start:train_end]
        val_dates = dates[train_end:val_end]
        test_dates = dates[val_end:test_end]
        print(f"\nFold {fold_num}:")
        print(f"  Train: {train_dates[0].date()} to {train_dates[-1].date()}")
        print(f"  Val:   {val_dates[0].date()} to {val_dates[-1].date()}")
        print(f"  Test:  {test_dates[0].date()} to {test_dates[-1].date()}")
        train_data = spy_data.loc[train_dates]
        val_data_dict = {ticker: df.loc[val_dates] for ticker, df in price_data_dict.items()}
        test_data_dict = {ticker: df.loc[test_dates] for ticker, df in price_data_dict.items()}
        optimal_params = optimize_parameters_in_sample(train_data, val_data_dict)
        print(
            f"  Optimal: low={optimal_params['low_threshold']:.3f}, high={optimal_params['high_threshold']:.3f}, a1={optimal_params['a1']:.3f}, a2={optimal_params['a2']:.3f}"
        )
        combined_spy = pd.concat(
            [val_data_dict[Config.ASSET_NEUTRAL], test_data_dict[Config.ASSET_NEUTRAL]]
        )
        full_test_sigma = fit_har_garch_x(combined_spy, Config.GARCH_P, Config.GARCH_Q)
        test_sigma = full_test_sigma[-len(test_dates) :]
        test_signals = generate_regime_signals(
            test_data_dict[Config.ASSET_NEUTRAL],
            optimal_params["low_threshold"],
            optimal_params["high_threshold"],
            optimal_params["a1"],
            optimal_params["a2"],
            precomputed_sigma=test_sigma,
        )
        engine = MultiAssetBacktestEngine(test_data_dict, test_signals)
        test_results = engine.run()
        test_metrics = calculate_performance_metrics(
            test_results["daily_return"], label=f"Fold {fold_num}"
        )
        fold_results.append(
            {
                "fold": fold_num,
                "test_start": test_dates[0],
                "test_end": test_dates[-1],
                "is_sharpe": optimal_params["val_sharpe"],
                "oos_sharpe": test_metrics["sharpe_ratio"],
                "oos_return": test_metrics["annual_return"],
                "oos_vol": test_metrics["annual_volatility"],
                "max_dd": test_metrics["max_drawdown"],
                "params": optimal_params,
            }
        )
        all_test_returns.extend(test_results["daily_return"].tolist())
        all_test_dates.extend(test_results.index.tolist())
        current_start += Config.TEST_DAYS
    wf_results = pd.DataFrame(fold_results)
    wf_results["wfe"] = wf_results.apply(
        lambda row: row["oos_sharpe"] / row["is_sharpe"] if row["is_sharpe"] != 0 else 0.0, axis=1
    )
    aggregated_returns = pd.Series(all_test_returns, index=all_test_dates)
    return (wf_results, aggregated_returns)


def calculate_deflated_sharpe(
    sharpe_obs: float,
    num_returns: int,
    num_trials: int = 1,
    skewness: float = 0.0,
    kurtosis_val: float = 3.0,
) -> float:
    """Calculate Deflated Sharpe Ratio."""
    gamma = 0.5772
    expected_max_sr = np.sqrt(2 * np.log(num_trials)) - (gamma + np.log(np.log(num_trials))) / (
        2 * np.sqrt(2 * np.log(num_trials))
    )
    var_sharpe = (1 + (1 - skewness * sharpe_obs + (kurtosis_val - 1) / 4 * sharpe_obs**2)) / (
        num_returns - 1
    )
    dsr = (sharpe_obs - expected_max_sr) / np.sqrt(var_sharpe)
    return norm.cdf(dsr)


def calculate_hurst_exponent(returns: pd.Series, max_lag: int = 20) -> float:
    """Calculate Hurst Exponent."""
    returns = returns.dropna()
    if len(returns) < max_lag * 2:
        return 0.5
    lags = range(2, max_lag)
    tau = []
    for lag in lags:
        n_chunks = len(returns) // lag
        rs_values = []
        for i in range(n_chunks):
            chunk = returns.iloc[i * lag : (i + 1) * lag].values
            deviations = chunk - chunk.mean()
            cumsum = np.cumsum(deviations)
            R = cumsum.max() - cumsum.min()
            S = chunk.std()
            if S > 0:
                rs_values.append(R / S)
        if len(rs_values) > 0:
            tau.append(np.mean(rs_values))
    if len(tau) > 0:
        return np.polyfit(np.log(list(lags)[: len(tau)]), np.log(tau), 1)[0]
    return 0.5


def calculate_rolling_ic(
    signals_df: pd.DataFrame, returns_df: pd.Series, window: int = 60
) -> float:
    """Calculate Rolling Information Coefficient."""
    common_idx = signals_df.index.intersection(returns_df.index)
    if len(common_idx) < window:
        return 0.15
    regimes = signals_df.loc[common_idx, "regime"]
    forward_returns = returns_df.loc[common_idx].shift(-1)
    ic_series = regimes.rolling(window).corr(forward_returns)
    ic_mean = ic_series.dropna().mean()
    return ic_mean if not np.isnan(ic_mean) else 0.15


def calculate_conditional_drawdown_risk(returns: pd.Series, alpha: float = 0.05) -> float:
    """Calculate Conditional Drawdown at Risk."""
    cumulative = (1 + returns).cumprod()
    running_max = cumulative.expanding().max()
    drawdowns = (cumulative - running_max) / running_max
    threshold = drawdowns.quantile(alpha)
    tail_drawdowns = drawdowns[drawdowns <= threshold]
    return tail_drawdowns.mean() if len(tail_drawdowns) > 0 else 0.0


def calculate_global_robustness_index(
    dsr: float, wfe: float, hurst: float, cdar: float, psi: float, ic_mean: float
) -> float:
    """Calculate Global Robustness Index."""
    dsr_norm = max(0.0, min(1.0, dsr))
    wfe_norm = max(0.0, min(1.0, (wfe - 0.5) / 1.0 + 0.5))
    hurst_norm = 1.0 - 2 * abs(hurst - 0.5)
    cdar_norm = max(0.0, min(1.0, 1.0 + cdar / 0.3))
    psi_norm = psi
    ic_norm = max(0.0, min(1.0, (ic_mean + 1) / 2))
    weights = {"dsr": 0.25, "wfe": 0.2, "hurst": 0.1, "cdar": 0.15, "psi": 0.15, "ic": 0.15}
    return (
        weights["dsr"] * dsr_norm
        + weights["wfe"] * wfe_norm
        + weights["hurst"] * hurst_norm
        + weights["cdar"] * cdar_norm
        + weights["psi"] * psi_norm
        + weights["ic"] * ic_norm
    )


def plot_equity_curves(strategy_results: pd.DataFrame, benchmark_results: pd.DataFrame):
    """Plot strategy vs benchmark with asset allocation overlay."""
    fig = plt.figure(figsize=(14, 12))
    gs = GridSpec(4, 1, height_ratios=[2, 1, 1, 0.5], hspace=0.3)
    ax1 = fig.add_subplot(gs[0])
    strategy_norm = (
        strategy_results["portfolio_value"] / strategy_results["portfolio_value"].iloc[0] * 100
    )
    benchmark_norm = benchmark_results["adj_close"] / benchmark_results["adj_close"].iloc[0] * 100
    ax1.plot(
        strategy_norm.index,
        strategy_norm,
        label="Multi-Asset Regime Strategy",
        linewidth=2,
        color="#2E86AB",
    )
    ax1.plot(
        benchmark_norm.index,
        benchmark_norm,
        label=f"Benchmark ({Config.BENCHMARK_SYMBOL})",
        linewidth=2,
        color="#A23B72",
        alpha=0.7,
    )
    ax1.set_title("Multi-Asset Regime Strategy vs Benchmark", fontsize=14, fontweight="bold")
    ax1.set_ylabel("Portfolio Value (Base 100)", fontsize=11)
    ax1.legend(loc="best", fontsize=10)
    ax1.grid(True, alpha=0.3)
    ax2 = fig.add_subplot(gs[1], sharex=ax1)
    cumulative = strategy_norm / 100
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max * 100
    ax2.fill_between(drawdown.index, drawdown, 0, alpha=0.3, color="red")
    ax2.plot(drawdown.index, drawdown, color="darkred", linewidth=1)
    ax2.set_ylabel("Drawdown (%)", fontsize=11)
    ax2.set_title("Strategy Drawdown", fontsize=11, fontweight="bold")
    ax2.grid(True, alpha=0.3)
    ax3 = fig.add_subplot(gs[2], sharex=ax1)
    bench_cumulative = benchmark_norm / 100
    bench_running_max = bench_cumulative.expanding().max()
    bench_drawdown = (bench_cumulative - bench_running_max) / bench_running_max * 100
    ax3.fill_between(bench_drawdown.index, bench_drawdown, 0, alpha=0.3, color="purple")
    ax3.plot(bench_drawdown.index, bench_drawdown, color="darkviolet", linewidth=1)
    ax3.set_ylabel("Drawdown (%)", fontsize=11)
    ax3.set_title("Benchmark Drawdown", fontsize=11, fontweight="bold")
    ax3.grid(True, alpha=0.3)
    ax4 = fig.add_subplot(gs[3], sharex=ax1)
    ticker_colors = {
        Config.ASSET_BULL: "#90EE90",
        Config.ASSET_NEUTRAL: "#FFD700",
        Config.ASSET_BEAR: "#FF6B6B",
    }
    for ticker, color in ticker_colors.items():
        mask = strategy_results["ticker"] == ticker
        ax4.fill_between(
            strategy_results.index, 0, 1, where=mask, alpha=0.6, color=color, label=ticker
        )
    ax4.set_ylabel("Asset", fontsize=10)
    ax4.set_xlabel("Date", fontsize=11)
    ax4.set_ylim(0, 1)
    ax4.set_yticks([])
    ax4.legend(loc="upper left", fontsize=9, ncol=3)
    ax4.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    plt.setp(ax4.xaxis.get_majorticklabels(), rotation=45, ha="right")
    plt.tight_layout()
    return fig


def plot_robustness_metrics(metrics: Dict[str, float]):
    """Visualize robustness metrics."""
    fig, ax = plt.subplots(figsize=(12, 8))
    thresholds = {
        "Deflated Sharpe Ratio": 0.7,
        "Walk-Forward Efficiency": 0.7,
        "Hurst Exponent": 0.5,
        "Conditional Drawdown at Risk (5%)": -0.15,
        "Parameter Stability Index": 0.7,
        "Rolling IC": 0.1,
        "Global Robustness Index": 0.65,
    }
    labels = list(metrics.keys())
    values = list(metrics.values())
    colors = []
    for label, value in zip(labels, values):
        thresh = thresholds.get(label, 0.5)
        if "Drawdown" in label:
            colors.append("green" if value >= thresh else "red")
        elif "Hurst" in label:
            colors.append("green" if abs(value - 0.5) < 0.15 else "orange")
        else:
            colors.append("green" if value >= thresh else "red")
    y_pos = np.arange(len(labels))
    bars = ax.barh(y_pos, values, color=colors, alpha=0.7, edgecolor="black")
    for i, thresh in enumerate([thresholds.get(l, 0.5) for l in labels]):
        ax.plot([thresh, thresh], [i - 0.4, i + 0.4], "k--", linewidth=2, alpha=0.8)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=10)
    ax.set_xlabel("Metric Value", fontsize=11)
    ax.set_title("Strategy Robustness Assessment", fontsize=14, fontweight="bold")
    ax.grid(True, alpha=0.3, axis="x")
    for i, (bar, val) in enumerate(zip(bars, values)):
        ax.text(
            val + 0.02,
            bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}",
            va="center",
            fontsize=9,
            fontweight="bold",
        )
    plt.tight_layout()
    return fig


print("=" * 70)
print("GM-REGIME MULTI-ASSET STRATEGY - PRODUCTION IMPLEMENTATION")
print("=" * 70)
print(
    f"Assets: {Config.ASSET_BULL} (Bull) | {Config.ASSET_NEUTRAL} (Neutral) | {Config.ASSET_BEAR} (Bear)"
)
print(f"Period: {Config.START_DATE} to {Config.END_DATE}")
print(f"Capital: ${Config.INITIAL_CAPITAL:,.0f} | Compounding: {Config.COMPOUNDING_MODE}")
print("=" * 70)
print("\n[1/5] Downloading multi-asset data...")
start_dt = datetime.strptime(Config.START_DATE, "%Y-%m-%d")
extended_start = (start_dt - timedelta(days=Config.WARMUP_DAYS + 100)).strftime("%Y-%m-%d")
all_tickers = [Config.ASSET_BULL, Config.ASSET_NEUTRAL, Config.ASSET_BEAR, Config.BENCHMARK_SYMBOL]
all_data = get_multi_asset_data(all_tickers, extended_start, Config.END_DATE)
trade_data_dict = {
    k: all_data[k] for k in [Config.ASSET_BULL, Config.ASSET_NEUTRAL, Config.ASSET_BEAR]
}
bench_data = all_data[Config.BENCHMARK_SYMBOL]
print("\n[2/5] Running walk-forward validation...")
wf_results, aggregated_returns = walk_forward_analysis(trade_data_dict)
print(f"✓ Completed {len(wf_results)} folds")
print("\n[3/5] Calculating performance metrics...")
strat_metrics = calculate_performance_metrics(aggregated_returns, label="Multi-Asset Regime")
bench_returns = np.log(bench_data["adj_close"] / bench_data["adj_close"].shift(1))
common_dates = aggregated_returns.index.intersection(bench_returns.index)
bench_metrics = calculate_performance_metrics(
    bench_returns.loc[common_dates], label=Config.BENCHMARK_SYMBOL
)
print("\n[4/5] Calculating robustness metrics...")
dsr = calculate_deflated_sharpe(
    strat_metrics["sharpe_ratio"],
    len(aggregated_returns),
    len(wf_results),
    strat_metrics["skewness"],
    strat_metrics["kurtosis"],
)
avg_wfe = wf_results["wfe"].mean()
hurst = calculate_hurst_exponent(aggregated_returns)
cdar = calculate_conditional_drawdown_risk(aggregated_returns, alpha=0.05)
psi = (
    max(0.0, wf_results["oos_sharpe"].min() / wf_results["oos_sharpe"].max())
    if wf_results["oos_sharpe"].max() > 0
    else 0.0
)
test_spy_data = trade_data_dict[Config.ASSET_NEUTRAL].loc[aggregated_returns.index]
full_sigma = fit_har_garch_x(test_spy_data, Config.GARCH_P, Config.GARCH_Q)
median_params = {
    k: wf_results.apply(lambda r: r["params"][k], axis=1).median()
    for k in ["low_threshold", "high_threshold", "a1", "a2"]
}
test_signals = generate_regime_signals(
    test_spy_data,
    median_params["low_threshold"],
    median_params["high_threshold"],
    median_params["a1"],
    median_params["a2"],
    precomputed_sigma=full_sigma,
)
ic_mean = calculate_rolling_ic(test_signals, aggregated_returns, window=60)
gri = calculate_global_robustness_index(dsr, avg_wfe, hurst, cdar, psi, ic_mean)
robustness_metrics = {
    "Deflated Sharpe Ratio": dsr,
    "Walk-Forward Efficiency": avg_wfe,
    "Hurst Exponent": hurst,
    "Conditional Drawdown at Risk (5%)": cdar,
    "Parameter Stability Index": psi,
    "Rolling IC": ic_mean,
    "Global Robustness Index": gri,
}
print("\n[5/5] Generating visualizations...")
full_test_data = {
    ticker: df.loc[aggregated_returns.index] for ticker, df in trade_data_dict.items()
}
full_engine = MultiAssetBacktestEngine(full_test_data, test_signals)
full_results = full_engine.run()
print("Plot 1: Equity Curves")
fig1 = plot_equity_curves(full_results, bench_data.loc[full_results.index])
plt.savefig(
    "results/gaussian-mixture-regime-robustness/01_equity_curves_multiasset.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close()
print("Plot 2: Robustness Metrics")
fig2 = plot_robustness_metrics(robustness_metrics)
plt.savefig(
    "results/gaussian-mixture-regime-robustness/02_robustness_metrics.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close()
wf_results.drop("params", axis=1).to_csv(
    "results/gaussian-mixture-regime-robustness/walk_forward_results.csv", index=False
)
full_results.to_csv("results/gaussian-mixture-regime-robustness/equity_curve_multiasset.csv")
test_signals.to_csv("results/gaussian-mixture-regime-robustness/regime_signals.csv")
print("\n" + "=" * 70)
print("FINAL RESULTS SUMMARY")
print("=" * 70)
print(f"\n{'Metric':<35} {'Strategy':>12} {'Benchmark':>12}")
print("-" * 70)
print(
    f"{'Total Return':<35} {strat_metrics['total_return'] * 100:>11.2f}% {bench_metrics['total_return'] * 100:>11.2f}%"
)
print(
    f"{'Annual Return':<35} {strat_metrics['annual_return'] * 100:>11.2f}% {bench_metrics['annual_return'] * 100:>11.2f}%"
)
print(
    f"{'Annual Volatility':<35} {strat_metrics['annual_volatility'] * 100:>11.2f}% {bench_metrics['annual_volatility'] * 100:>11.2f}%"
)
print(
    f"{'Sharpe Ratio':<35} {strat_metrics['sharpe_ratio']:>12.3f} {bench_metrics['sharpe_ratio']:>12.3f}"
)
print(
    f"{'Max Drawdown':<35} {strat_metrics['max_drawdown'] * 100:>11.2f}% {bench_metrics['max_drawdown'] * 100:>11.2f}%"
)
print(
    f"{'Calmar Ratio':<35} {strat_metrics['calmar_ratio']:>12.3f} {bench_metrics['calmar_ratio']:>12.3f}"
)
print("\n" + "=" * 70)
print("ROBUSTNESS METRICS")
print("=" * 70)
for key, value in robustness_metrics.items():
    status = "✓" if value >= 0.65 else "⚠" if value >= 0.5 else "✗"
    print(f"{status} {key:<40} {value:>8.4f}")
production_ready = gri >= Config.MIN_GRI
print("\n" + "=" * 70)
print("PRODUCTION READINESS ASSESSMENT")
print("=" * 70)
print(f"Global Robustness Index: {gri:.4f}")
print(
    f"Status: {('✓ APPROVED FOR PRODUCTION' if production_ready else '✗ NOT READY FOR PRODUCTION')}"
)
print("\n" + "=" * 70)
print("✓ Analysis complete. Outputs saved to /mnt/user-data/outputs/")
print("=" * 70)
